# Enhanced VLM Candidate Classifier

This notebook upgrades the first binary candidate classifier into an **episode-level 6-way selector**.

## Why this version
The previous BCE setup treated each candidate independently, which caused the model to collapse toward predicting all negatives because the data has a 1:5 positive:negative ratio inside each episode.

## Main change
For each episode:
- one image
- one instruction
- six candidate texts
- exactly one correct candidate index

The model produces 6 scores and trains with **cross entropy**.

## Backbone choice
- Default: CLIP (`openai/clip-vit-base-patch32`)
- Optional: SigLIP (`google/siglip-base-patch16-224`)
- Keep the backbone frozen for now.
- QLoRA is still not the right tool here because we are not fine-tuning a large generative VLM.

In [1]:
# Optional install cell if transformers is missing.
# import sys
# !{sys.executable} -m pip install transformers accelerate

In [2]:
from pathlib import Path
import json
import random
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoModel, AutoProcessor

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
PROJECT_ROOT = Path('/home/gyanig/catkin_ws/src/tabletop_workspace_opt')
DATA_ROOT = PROJECT_ROOT / 'data' / 'milk_candidate_cls'
SAMPLES_PATH = DATA_ROOT / 'candidate_samples.jsonl'
EPISODES_PATH = DATA_ROOT / 'episodes.jsonl'
SPLIT_ROOT = DATA_ROOT / 'splits'
SPLIT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'openai/clip-vit-base-patch32'  # Try 'google/siglip-base-patch16-224' after CLIP works.
BATCH_SIZE = 4
EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
assert SAMPLES_PATH.exists(), SAMPLES_PATH

In [4]:
def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

df = pd.DataFrame(load_jsonl(SAMPLES_PATH))
df.head(2)

,sample_id,episode_id,scene_id,view_id,image_path,instruction,correct_candidate_id,label,task_type,scene_notes,episode_notes,slot_assignment,candidate_id,object_short,object_name,grasp_type,grasp_description,task_suitability,candidate_text
0,ep_milk_scene_01_top_1__whole_top,ep_milk_scene_01_top_1,milk_scene_01_top,top,images/milk_scene_01_top.png,Pick up the whole milk.,whole_top,1,pickup,fixed layout v1,,"{'left': 'whole_milk', 'center': 'oat_milk', '...",whole_top,whole,whole_milk,top,top grasp,pickup,Object: whole_milk. Grasp: top grasp. Task sui...
1,ep_milk_scene_01_top_1__whole_side,ep_milk_scene_01_top_1,milk_scene_01_top,top,images/milk_scene_01_top.png,Pick up the whole milk.,whole_top,0,pickup,fixed layout v1,,"{'left': 'whole_milk', 'center': 'oat_milk', '...",whole_side,whole,whole_milk,side,side grasp,pour,Object: whole_milk. Grasp: side grasp. Task su...


In [5]:
def base_scene_id(scene_id: str) -> str:
    for suffix in ('_top', '_side', '_lean'):
        if scene_id.endswith(suffix):
            return scene_id[: -len(suffix)]
    return scene_id

df['base_scene_id'] = df['scene_id'].map(base_scene_id)
df['text_input'] = df.apply(
    lambda row: f"Instruction: {row['instruction']}\nCandidate: {row['candidate_text']}", axis=1
)
print('rows:', len(df))
print('episodes:', df['episode_id'].nunique())
print('base scenes:', sorted(df['base_scene_id'].unique()))

rows: 258
episodes: 43
base scenes: ['milk_scene_01', 'milk_scene_02']


In [6]:
base_scenes = sorted(df['base_scene_id'].unique())
assert len(base_scenes) >= 2, 'Need at least two base scenes for scene-level split.'

train_scenes = base_scenes[:-1]
val_scenes = base_scenes[-1:]

train_df = df[df['base_scene_id'].isin(train_scenes)].reset_index(drop=True)
val_df = df[df['base_scene_id'].isin(val_scenes)].reset_index(drop=True)

print('train scenes:', train_scenes)
print('val scenes  :', val_scenes)
print('train rows   :', len(train_df))
print('val rows     :', len(val_df))

train scenes: ['milk_scene_01']
val scenes  : ['milk_scene_02']
train rows   : 126
val rows     : 132


In [7]:
def build_episode_records(frame: pd.DataFrame):
    records = []
    grouped = frame.groupby('episode_id', sort=True)
    for episode_id, g in grouped:
        g = g.sort_values('candidate_id').reset_index(drop=True)
        assert len(g) == 6, f'{episode_id} has {len(g)} candidates instead of 6'
        positive_rows = g[g['label'] == 1]
        assert len(positive_rows) == 1, f'{episode_id} must have exactly one positive candidate'
        target_index = int(positive_rows.index[0])
        records.append({
            'episode_id': episode_id,
            'scene_id': g.iloc[0]['scene_id'],
            'base_scene_id': g.iloc[0]['base_scene_id'],
            'view_id': g.iloc[0]['view_id'],
            'image_path': g.iloc[0]['image_path'],
            'instruction': g.iloc[0]['instruction'],
            'candidate_ids': g['candidate_id'].tolist(),
            'candidate_texts': g['text_input'].tolist(),
            'correct_candidate_id': positive_rows.iloc[0]['candidate_id'],
            'target_index': target_index,
        })
    return records

train_records = build_episode_records(train_df)
val_records = build_episode_records(val_df)

print('train episodes:', len(train_records))
print('val episodes  :', len(val_records))
train_records[0]

train episodes: 21
val episodes  : 22


{'episode_id': 'ep_milk_scene_01_lean_1',
 'scene_id': 'milk_scene_01_lean',
 'base_scene_id': 'milk_scene_01',
 'view_id': 'lean',
 'image_path': 'images/milk_scene_01_lean.png',
 'instruction': 'Pick up the whole milk.',
 'candidate_ids': ['oat_side',
  'oat_top',
  'soy_side',
  'soy_top',
  'whole_side',
  'whole_top'],
 'candidate_texts': ['Instruction: Pick up the whole milk.\nCandidate: Object: oat_milk. Grasp: side grasp. Task suitability: pour.',
  'Instruction: Pick up the whole milk.\nCandidate: Object: oat_milk. Grasp: top grasp. Task suitability: pickup.',
  'Instruction: Pick up the whole milk.\nCandidate: Object: soy_milk. Grasp: side grasp. Task suitability: pour.',
  'Instruction: Pick up the whole milk.\nCandidate: Object: soy_milk. Grasp: top grasp. Task suitability: pickup.',
  'Instruction: Pick up the whole milk.\nCandidate: Object: whole_milk. Grasp: side grasp. Task suitability: pour.',
  'Instruction: Pick up the whole milk.\nCandidate: Object: whole_milk. Gras

In [8]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
backbone = AutoModel.from_pretrained(MODEL_NAME).to(device)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False

def get_image_features(model, pixel_values):
    if hasattr(model, 'get_image_features'):
        return model.get_image_features(pixel_values=pixel_values)
    vision_outputs = model.vision_model(pixel_values=pixel_values)
    return vision_outputs.pooler_output

def get_text_features(model, input_ids, attention_mask):
    if hasattr(model, 'get_text_features'):
        return model.get_text_features(input_ids=input_ids, attention_mask=attention_mask)
    text_outputs = model.text_model(input_ids=input_ids, attention_mask=attention_mask)
    return text_outputs.pooler_output

def l2_normalize(x):
    return x / x.norm(dim=-1, keepdim=True).clamp_min(1e-6)

with torch.no_grad():
    dummy_img = Image.open(DATA_ROOT / train_records[0]['image_path']).convert('RGB')
    dummy_text = train_records[0]['candidate_texts'][0]
    dummy = processor(images=dummy_img, text=dummy_text, return_tensors='pt', padding=True)
    embed_dim = int(get_image_features(backbone, dummy['pixel_values'].to(device)).shape[-1])
embed_dim

512

In [9]:
class EpisodeDataset(Dataset):
    def __init__(self, records, image_root):
        self.records = records
        self.image_root = image_root

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        image = Image.open(self.image_root / rec['image_path']).convert('RGB')
        return {
            'image': image,
            'candidate_texts': rec['candidate_texts'],
            'target_index': rec['target_index'],
            'episode_id': rec['episode_id'],
            'candidate_ids': rec['candidate_ids'],
            'correct_candidate_id': rec['correct_candidate_id'],
        }

def collate_episode(batch):
    images = [item['image'] for item in batch]
    # Flatten text so processor can tokenize them together.
    texts = []
    for item in batch:
        texts.extend(item['candidate_texts'])

    image_inputs = processor(images=images, return_tensors='pt')
    text_inputs = processor(text=texts, return_tensors='pt', padding=True, truncation=True)

    return {
        'pixel_values': image_inputs['pixel_values'],
        'input_ids': text_inputs['input_ids'],
        'attention_mask': text_inputs['attention_mask'],
        'targets': torch.tensor([item['target_index'] for item in batch], dtype=torch.long),
        'episode_ids': [item['episode_id'] for item in batch],
        'candidate_ids': [item['candidate_ids'] for item in batch],
        'correct_candidate_ids': [item['correct_candidate_id'] for item in batch],
        'num_candidates': len(batch[0]['candidate_texts']),
    }

train_ds = EpisodeDataset(train_records, DATA_ROOT)
val_ds = EpisodeDataset(val_records, DATA_ROOT)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_episode)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_episode)

In [10]:
class EpisodeRanker(nn.Module):
    def __init__(self, vlm_backbone, embed_dim):
        super().__init__()
        self.vlm_backbone = vlm_backbone
        self.scorer = nn.Sequential(
            nn.Linear(embed_dim * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1),
        )

    def forward(self, pixel_values, input_ids, attention_mask, num_candidates):
        batch_size = pixel_values.shape[0]
        with torch.no_grad():
            image_feat = get_image_features(self.vlm_backbone, pixel_values)
            text_feat = get_text_features(self.vlm_backbone, input_ids, attention_mask)
            image_feat = l2_normalize(image_feat)
            text_feat = l2_normalize(text_feat)

        image_feat = image_feat.unsqueeze(1).expand(batch_size, num_candidates, image_feat.shape[-1])
        text_feat = text_feat.view(batch_size, num_candidates, -1)
        fused = torch.cat([image_feat, text_feat, image_feat * text_feat], dim=-1)
        logits = self.scorer(fused).squeeze(-1)
        return logits

model = EpisodeRanker(backbone, embed_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.scorer.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

In [11]:
def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    num_items = 0
    y_true = []
    y_pred = []

    for batch in loader:
        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['targets'].to(device)
        num_candidates = batch['num_candidates']

        logits = model(pixel_values, input_ids, attention_mask, num_candidates)
        loss = criterion(logits, targets)

        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        pred = logits.argmax(dim=1)
        y_true.extend(targets.detach().cpu().tolist())
        y_pred.extend(pred.detach().cpu().tolist())
        total_loss += float(loss.item()) * len(targets)
        num_items += len(targets)

    top1_acc = float(np.mean([int(a == b) for a, b in zip(y_true, y_pred)])) if y_true else 0.0
    return {
        'loss': total_loss / max(num_items, 1),
        'episode_top1': top1_acc,
    }

In [12]:
history = []
best_val = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    row = {
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_episode_top1': train_metrics['episode_top1'],
        'val_loss': val_metrics['loss'],
        'val_episode_top1': val_metrics['episode_top1'],
    }
    history.append(row)
    print(row)
    if row['val_episode_top1'] > best_val:
        best_val = row['val_episode_top1']
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

history_df = pd.DataFrame(history)
history_df

{'epoch': 1, 'train_loss': 1.7883862200237455, 'train_episode_top1': 0.09523809523809523, 'val_loss': 1.778737328269265, 'val_episode_top1': 0.4090909090909091}
{'epoch': 2, 'train_loss': 1.7768675826844715, 'train_episode_top1': 0.19047619047619047, 'val_loss': 1.7642470706592908, 'val_episode_top1': 0.3181818181818182}
{'epoch': 3, 'train_loss': 1.7589631080627441, 'train_episode_top1': 0.3333333333333333, 'val_loss': 1.7488423911007969, 'val_episode_top1': 0.45454545454545453}
{'epoch': 4, 'train_loss': 1.745063793091547, 'train_episode_top1': 0.23809523809523808, 'val_loss': 1.7299221645702014, 'val_episode_top1': 0.45454545454545453}
{'epoch': 5, 'train_loss': 1.7232750143323625, 'train_episode_top1': 0.38095238095238093, 'val_loss': 1.710781921039928, 'val_episode_top1': 0.45454545454545453}
{'epoch': 6, 'train_loss': 1.7110941353298368, 'train_episode_top1': 0.3333333333333333, 'val_loss': 1.68916316465898, 'val_episode_top1': 0.45454545454545453}
{'epoch': 7, 'train_loss': 1.68

,epoch,train_loss,train_episode_top1,val_loss,val_episode_top1
0,1,1.788386,0.095238,1.778737,0.409091
1,2,1.776868,0.190476,1.764247,0.318182
2,3,1.758963,0.333333,1.748842,0.454545
3,4,1.745064,0.238095,1.729922,0.454545
4,5,1.723275,0.380952,1.710782,0.454545
5,6,1.711094,0.333333,1.689163,0.454545
6,7,1.687629,0.285714,1.666575,0.454545
7,8,1.665681,0.333333,1.645382,0.454545
8,9,1.642533,0.380952,1.621228,0.454545
9,10,1.607599,0.428571,1.596232,0.454545


In [13]:
if best_state is not None:
    model.load_state_dict(best_state)

save_dir = PROJECT_ROOT / 'outputs' / 'vlm_candidate_classifier_enhanced'
save_dir.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        'model_name': MODEL_NAME,
        'scorer_state_dict': model.scorer.state_dict(),
        'embed_dim': embed_dim,
        'train_scenes': train_scenes,
        'val_scenes': val_scenes,
    },
    save_dir / 'episode_ranker.pt'
)
history_df.to_csv(save_dir / 'history.csv', index=False)
history_df

,epoch,train_loss,train_episode_top1,val_loss,val_episode_top1
0,1,1.788386,0.095238,1.778737,0.409091
1,2,1.776868,0.190476,1.764247,0.318182
2,3,1.758963,0.333333,1.748842,0.454545
3,4,1.745064,0.238095,1.729922,0.454545
4,5,1.723275,0.380952,1.710782,0.454545
5,6,1.711094,0.333333,1.689163,0.454545
6,7,1.687629,0.285714,1.666575,0.454545
7,8,1.665681,0.333333,1.645382,0.454545
8,9,1.642533,0.380952,1.621228,0.454545
9,10,1.607599,0.428571,1.596232,0.454545


## Text-only baseline

This baseline removes the image entirely and ranks the 6 candidates using only instruction and candidate text.

In [14]:
class TextOnlyEpisodeDataset(Dataset):
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        rec = self.records[idx]
        return {
            'candidate_texts': rec['candidate_texts'],
            'target_index': rec['target_index'],
            'episode_id': rec['episode_id'],
            'candidate_ids': rec['candidate_ids'],
            'correct_candidate_id': rec['correct_candidate_id'],
        }
def collate_text_only(batch):
    texts = []
    for item in batch:
        texts.extend(item['candidate_texts'])
    text_inputs = processor(text=texts, return_tensors='pt', padding=True, truncation=True)
    return {
        'input_ids': text_inputs['input_ids'],
        'attention_mask': text_inputs['attention_mask'],
        'targets': torch.tensor([item['target_index'] for item in batch], dtype=torch.long),
        'episode_ids': [item['episode_id'] for item in batch],
        'candidate_ids': [item['candidate_ids'] for item in batch],
        'correct_candidate_ids': [item['correct_candidate_id'] for item in batch],
        'num_candidates': len(batch[0]['candidate_texts']),
    }
text_train_ds = TextOnlyEpisodeDataset(train_records)
text_val_ds = TextOnlyEpisodeDataset(val_records)
text_train_loader = DataLoader(text_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_text_only)
text_val_loader = DataLoader(text_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_text_only)
class TextOnlyEpisodeRanker(nn.Module):
    def __init__(self, vlm_backbone, embed_dim):
        super().__init__()
        self.vlm_backbone = vlm_backbone
        self.scorer = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1),
        )
    def forward(self, input_ids, attention_mask, num_candidates):
        batch_size = input_ids.shape[0] // num_candidates
        with torch.no_grad():
            text_feat = get_text_features(self.vlm_backbone, input_ids, attention_mask)
            text_feat = l2_normalize(text_feat)
        text_feat = text_feat.view(batch_size, num_candidates, -1)
        return self.scorer(text_feat).squeeze(-1)
text_model = TextOnlyEpisodeRanker(backbone, embed_dim).to(device)
text_criterion = nn.CrossEntropyLoss()
text_optimizer = torch.optim.AdamW(text_model.scorer.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

In [15]:
def run_text_epoch(loader, train=True):
    text_model.train(train)
    total_loss = 0.0
    n = 0
    correct = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['targets'].to(device)
        logits = text_model(input_ids, attention_mask, batch['num_candidates'])
        loss = text_criterion(logits, targets)
        if train:
            text_optimizer.zero_grad()
            loss.backward()
            text_optimizer.step()
        pred = logits.argmax(dim=1)
        correct += int((pred == targets).sum().item())
        n += len(targets)
        total_loss += float(loss.item()) * len(targets)
    return {'loss': total_loss / max(n, 1), 'episode_top1': correct / max(n, 1)}
text_history = []
best_text_val = -1.0
for epoch in range(1, EPOCHS + 1):
    train_metrics = run_text_epoch(text_train_loader, train=True)
    val_metrics = run_text_epoch(text_val_loader, train=False)
    row = {
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_episode_top1': train_metrics['episode_top1'],
        'val_loss': val_metrics['loss'],
        'val_episode_top1': val_metrics['episode_top1'],
    }
    text_history.append(row)
    print(row)
    best_text_val = max(best_text_val, row['val_episode_top1'])
text_history_df = pd.DataFrame(text_history)
print('Best text-only val top1:', best_text_val)
text_history_df

{'epoch': 1, 'train_loss': 1.786274030095055, 'train_episode_top1': 0.19047619047619047, 'val_loss': 1.7763657786629417, 'val_episode_top1': 0.45454545454545453}
{'epoch': 2, 'train_loss': 1.7700427713848295, 'train_episode_top1': 0.42857142857142855, 'val_loss': 1.7628829045729204, 'val_episode_top1': 0.45454545454545453}
{'epoch': 3, 'train_loss': 1.7574844587416876, 'train_episode_top1': 0.42857142857142855, 'val_loss': 1.745498462156816, 'val_episode_top1': 0.45454545454545453}
{'epoch': 4, 'train_loss': 1.7414082345508395, 'train_episode_top1': 0.42857142857142855, 'val_loss': 1.7260455001484265, 'val_episode_top1': 0.45454545454545453}
{'epoch': 5, 'train_loss': 1.724054597672962, 'train_episode_top1': 0.23809523809523808, 'val_loss': 1.7066202272068371, 'val_episode_top1': 0.45454545454545453}
{'epoch': 6, 'train_loss': 1.6997085412343342, 'train_episode_top1': 0.42857142857142855, 'val_loss': 1.6818390976298938, 'val_episode_top1': 0.45454545454545453}
{'epoch': 7, 'train_loss'

,epoch,train_loss,train_episode_top1,val_loss,val_episode_top1
0,1,1.786274,0.190476,1.776366,0.454545
1,2,1.770043,0.428571,1.762883,0.454545
2,3,1.757484,0.428571,1.745498,0.454545
3,4,1.741408,0.428571,1.726046,0.454545
4,5,1.724055,0.238095,1.706620,0.454545
5,6,1.699709,0.428571,1.681839,0.454545
6,7,1.679234,0.333333,1.654336,0.454545
7,8,1.656123,0.380952,1.625963,0.454545
8,9,1.622027,0.380952,1.597275,0.454545
9,10,1.589277,0.380952,1.568097,0.454545


## SigLIP comparison

This reruns the same episode-level image+text ranker with a SigLIP backbone so you can compare CLIP vs SigLIP directly.

In [18]:
import sys
!{sys.executable} -m pip install --upgrade protobuf==3.20.3


SIGLIP_MODEL_NAME = 'google/siglip-base-patch16-224'

siglip_processor = AutoProcessor.from_pretrained(SIGLIP_MODEL_NAME)
siglip_backbone = AutoModel.from_pretrained(SIGLIP_MODEL_NAME).to(device)
siglip_backbone.eval()
for p in siglip_backbone.parameters():
    p.requires_grad = False

with torch.no_grad():
    dummy_img = Image.open(DATA_ROOT / train_records[0]['image_path']).convert('RGB')
    dummy_text = train_records[0]['candidate_texts'][0]
    dummy = siglip_processor(images=dummy_img, text=dummy_text, return_tensors='pt', padding=True)
    siglip_embed_dim = int(get_image_features(siglip_backbone, dummy['pixel_values'].to(device)).shape[-1])

def collate_episode_siglip(batch):
    images = [item['image'] for item in batch]
    texts = []
    for item in batch:
        texts.extend(item['candidate_texts'])

    image_inputs = siglip_processor(images=images, return_tensors='pt')
    text_inputs = siglip_processor(text=texts, return_tensors='pt', padding=True, truncation=True)

    out = {
        'pixel_values': image_inputs['pixel_values'],
        'input_ids': text_inputs['input_ids'],
        'targets': torch.tensor([item['target_index'] for item in batch], dtype=torch.long),
        'num_candidates': len(batch[0]['candidate_texts']),
    }

    if 'attention_mask' in text_inputs:
        out['attention_mask'] = text_inputs['attention_mask']

    return out


siglip_train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_episode_siglip)
siglip_val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_episode_siglip)

siglip_model = EpisodeRanker(siglip_backbone, siglip_embed_dim).to(device)
siglip_criterion = nn.CrossEntropyLoss()
siglip_optimizer = torch.optim.AdamW(siglip_model.scorer.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable


In [20]:
def run_siglip_epoch(loader, train=True):
    siglip_model.train(train)
    total_loss = 0.0
    n = 0
    correct = 0

    for batch in loader:
        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch.get('attention_mask')
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)

        targets = batch['targets'].to(device)

        logits = siglip_model(pixel_values, input_ids, attention_mask, batch['num_candidates'])
        loss = siglip_criterion(logits, targets)

        if train:
            siglip_optimizer.zero_grad()
            loss.backward()
            siglip_optimizer.step()

        pred = logits.argmax(dim=1)
        correct += int((pred == targets).sum().item())
        n += len(targets)
        total_loss += float(loss.item()) * len(targets)

    return {
        'loss': total_loss / max(n, 1),
        'episode_top1': correct / max(n, 1),
    }


siglip_history = []
best_siglip_val = -1.0
for epoch in range(1, EPOCHS + 1):
    train_metrics = run_siglip_epoch(siglip_train_loader, train=True)
    val_metrics = run_siglip_epoch(siglip_val_loader, train=False)
    row = {
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_episode_top1': train_metrics['episode_top1'],
        'val_loss': val_metrics['loss'],
        'val_episode_top1': val_metrics['episode_top1'],
    }
    siglip_history.append(row)
    print(row)
    best_siglip_val = max(best_siglip_val, row['val_episode_top1'])

siglip_history_df = pd.DataFrame(siglip_history)
print('Best SigLIP val top1:', best_siglip_val)
siglip_history_df

{'epoch': 1, 'train_loss': 1.789514496212914, 'train_episode_top1': 0.19047619047619047, 'val_loss': 1.780567927794023, 'val_episode_top1': 0.6363636363636364}
{'epoch': 2, 'train_loss': 1.7763757024492537, 'train_episode_top1': 0.47619047619047616, 'val_loss': 1.7624950192191384, 'val_episode_top1': 0.7727272727272727}
{'epoch': 3, 'train_loss': 1.756629086676098, 'train_episode_top1': 0.42857142857142855, 'val_loss': 1.74430682442405, 'val_episode_top1': 0.8181818181818182}
{'epoch': 4, 'train_loss': 1.7355317274729412, 'train_episode_top1': 0.7142857142857143, 'val_loss': 1.7204598080028186, 'val_episode_top1': 0.8636363636363636}
{'epoch': 5, 'train_loss': 1.7003004494167508, 'train_episode_top1': 0.6666666666666666, 'val_loss': 1.6898281140760942, 'val_episode_top1': 0.9090909090909091}
{'epoch': 6, 'train_loss': 1.6707710425059001, 'train_episode_top1': 0.7619047619047619, 'val_loss': 1.6439292972738093, 'val_episode_top1': 0.9090909090909091}
{'epoch': 7, 'train_loss': 1.6439087

,epoch,train_loss,train_episode_top1,val_loss,val_episode_top1
0,1,1.789514,0.190476,1.780568,0.636364
1,2,1.776376,0.476190,1.762495,0.772727
2,3,1.756629,0.428571,1.744307,0.818182
3,4,1.735532,0.714286,1.720460,0.863636
4,5,1.700300,0.666667,1.689828,0.909091
5,6,1.670771,0.761905,1.643929,0.909091
6,7,1.643909,0.714286,1.588765,0.909091
7,8,1.576221,0.714286,1.536365,0.772727
8,9,1.506083,0.761905,1.475116,0.772727
9,10,1.487011,0.761905,1.399556,0.772727


In [21]:
comparison_df = pd.DataFrame({
    'model': ['image_text_clip', 'text_only_clip', 'image_text_siglip'],
    'best_val_episode_top1': [
        history_df['val_episode_top1'].max(),
        text_history_df['val_episode_top1'].max(),
        siglip_history_df['val_episode_top1'].max(),
    ],
})
comparison_df

,model,best_val_episode_top1
0,image_text_clip,0.590909
1,text_only_clip,0.863636
2,image_text_siglip,0.909091


## What to compare next

1. Rule-based baseline
2. Text-only baseline
3. Frozen CLIP episode ranker
4. Frozen SigLIP episode ranker

If CLIP and SigLIP are similar, keep the simpler one for the main report and use the other as an ablation.

Today's result:
Random: 0.167
Image+Text CLIP: ~0.59
Text-only CLIP: ~0.86
Image+Text SigLIP: ~0.91